# LLM Fine-Tuning Pipeline — Google Colab Runner

Runs the same pipeline as the Docker workflow (`data_pipeline.py` → `train.py` → `evaluate.py`),
for anyone without a local NVIDIA GPU. Run cells top to bottom on a **GPU runtime**
(Runtime > Change runtime type > GPU).

Docker remains the primary reproducible workflow for evaluators with an NVIDIA GPU — this
notebook is an additional path, not a replacement.

In [ ]:
PROJECT_DIR = "/content/llm-finetuning-pipeline"
REPO_URL = "https://github.com/Kanumilli-Poojitha/LLM-Finetuning-Pipeline.git"  # set to your repo

FORCE_RETRAIN = False   # set True to redo full training even if outputs/final_adapter exists
FORCE_REEVAL = False    # set True to redo evaluation even if outputs/eval_results exists

## 1. GPU check
Stops immediately if no GPU runtime is enabled.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected. Go to Runtime > Change runtime type > Hardware "
        "accelerator > GPU, then Runtime > Restart session and rerun this notebook."
    )

props = torch.cuda.get_device_properties(0)
print(f"GPU name: {torch.cuda.get_device_name(0)}")
print(f"CUDA version (torch build): {torch.version.cuda}")
print(f"Available VRAM: {props.total_memory / (1024**3):.1f} GB")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"torch.cuda.get_device_name(): {torch.cuda.get_device_name(0)}")

!nvidia-smi

## 2. Clone or update the repository
Skips cloning if the repo is already present; pulls latest changes instead.

In [ ]:
import os
import subprocess


def run(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)


if os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
    print(f"Repo already present at {PROJECT_DIR} — pulling latest changes.")
    run(f"git -C {PROJECT_DIR} pull --ff-only")
elif os.path.isdir(PROJECT_DIR):
    print(f"{PROJECT_DIR} exists but is not a git repo — using it as-is.")
else:
    run(f"git clone {REPO_URL} {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

## 3. Install dependencies
Checks installed versions against `requirements.txt` and only installs what's missing or mismatched. `torch`/`torchvision`/`torchaudio` are intentionally left alone — Colab ships a CUDA-matched build, same reasoning as the Dockerfile.

In [ ]:
import re
import importlib.metadata as ilm


def parse_requirements(path="requirements.txt"):
    pinned = {}
    with open(path) as f:
        for line in f:
            match = re.match(r"^([A-Za-z0-9_.\-]+)==([A-Za-z0-9_.\-]+)$", line.strip())
            if match:
                pinned[match.group(1).lower()] = match.group(2)
    return pinned


pinned = parse_requirements()
to_install = []
for pkg, version in pinned.items():
    try:
        installed = ilm.version(pkg)
    except ilm.PackageNotFoundError:
        installed = None
    status = "OK" if installed == version else ("MISSING" if installed is None else f"MISMATCH ({installed})")
    print(f"{pkg:<16} required={version:<10} installed={installed or '-':<10} [{status}]")
    if installed != version:
        to_install.append(f"{pkg}=={version}")

if to_install:
    print("\nInstalling/upgrading:", ", ".join(to_install))
    run(f"pip install --no-cache-dir {' '.join(to_install)}")
else:
    print("\nAll pinned dependencies already satisfied — nothing to install.")

Sanity-check the core stack imports and report versions — mirrors the same check the Dockerfile runs at build time.

In [ ]:
import accelerate
import bitsandbytes
import peft
import transformers
import trl
from trl import SFTTrainer

print("torch          ", torch.__version__)
print("transformers   ", transformers.__version__)
print("peft           ", peft.__version__)
print("trl            ", trl.__version__)
print("bitsandbytes   ", bitsandbytes.__version__)
print("accelerate     ", accelerate.__version__)
print("SFTTrainer import OK")
print("Import check OK — core stack is compatible.")

## 4. Hugging Face authentication
Tries Colab Secrets first (`HF_TOKEN`), falls back to a hidden `getpass()` prompt. Not required for the default public model (`microsoft/Phi-3-mini-4k-instruct`).

In [ ]:
from getpass import getpass

from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token:
    hf_token = getpass("Hugging Face token (leave blank to skip): ")

if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face Hub.")
else:
    print("No token provided — continuing without auth.")

## 5. Prepare the dataset
Skips if the processed splits already exist.

In [ ]:
processed_files = [f"data/processed/{split}.jsonl" for split in ("train", "validation", "test")]

if all(os.path.exists(p) for p in processed_files):
    print("Processed splits already exist — skipping (delete data/processed/ to regenerate).")
else:
    run("python -m src.data_pipeline --config configs/training_config.yaml")

## 6. Smoke test (20 steps)
Writes to `outputs/smoke_test`, a disposable directory separate from the real checkpoints — safe to rerun anytime.

In [ ]:
run(
    "python -m src.train --config configs/training_config.yaml "
    "--max_steps 20 --output_dir outputs/smoke_test"
)

## 7. Full training
Skips if a final adapter already exists, unless `FORCE_RETRAIN = True` above.

In [ ]:
final_adapter_dir = "outputs/final_adapter"
has_adapter = any(
    os.path.exists(os.path.join(final_adapter_dir, name))
    for name in ("adapter_model.safetensors", "adapter_model.bin")
)

if has_adapter and not FORCE_RETRAIN:
    print(f"{final_adapter_dir} already has trained weights — skipping. Set FORCE_RETRAIN = True to redo it.")
else:
    run("python -m src.train --config configs/training_config.yaml")

## 8. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/checkpoints/runs

## 9. Evaluation
Skips if results already exist, unless `FORCE_REEVAL = True` above. `evaluate.py` itself never overwrites an existing `evaluation_report.md`.

In [ ]:
metrics_path = "outputs/eval_results/metrics.json"

if os.path.exists(metrics_path) and not FORCE_REEVAL:
    print(f"{metrics_path} already exists — skipping. Set FORCE_REEVAL = True to redo it.")
else:
    run("python -m src.evaluate --config configs/training_config.yaml")

## 10. Zip and download outputs
Bundles the final adapter, eval results, TensorBoard logs, and the evaluation report.

In [ ]:
import shutil

from google.colab import files

staging = "/content/llm_finetuning_outputs"
os.makedirs(staging, exist_ok=True)

targets = [
    ("outputs/final_adapter", f"{staging}/final_adapter"),
    ("outputs/eval_results", f"{staging}/eval_results"),
    ("outputs/checkpoints/runs", f"{staging}/tensorboard_logs"),
    ("evaluation_report.md", f"{staging}/evaluation_report.md"),
]

for src, dst in targets:
    if not os.path.exists(src):
        print(f"Skipping missing: {src}")
        continue
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

archive_path = shutil.make_archive("/content/llm_finetuning_outputs", "zip", staging)
print("Archive ready:", archive_path)
files.download(archive_path)